In [5]:
import pandas as pd
import numpy as np
import os

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

folder = "/content/drive/MyDrive/AI_Crime_Prediction/datasets"

print("Files in datasets folder:\n")

for file in os.listdir(folder):
    print(file)

Files in datasets folder:

Bengaluru_Cleaned_Final.csv
Bengaluru_Spatial.csv
AI_Dataset.csv
Weekly_Risk_Dataset.csv
X_train.npy
y_train.npy
y_test.npy
X_test.npy
Bengaluru_Hawkes.csv
GCN_Node_Features.csv
GCN_Edges.csv
Test_2024_Jan_Mar.csv
LSTM_Sequences.csv


In [6]:
DATASET_PATH = "/content/drive/MyDrive/AI_Crime_Prediction/datasets/AI_Dataset.csv"

df = pd.read_csv(DATASET_PATH)

print(df.shape)
df.head()

(147769, 41)


/tmp/ipykernel_6476/328838031.py:3: DtypeWarning: Columns (15) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(DATASET_PATH)


,District_Name,UnitName,FIR_YEAR,FIR_MONTH,Offence_Duration,FIR_Day,FIR Type,FIR_Stage,Complaint_Mode,CrimeGroup_Name,...,Accused_ChargeSheeted Count,Conviction Count,Unit_ID,Date,DayOfWeek,IsWeekend,Quarter,Grid_X,Grid_Y,Grid_ID
0,Bengaluru City,Byadarahalli PS,2016,10,0,27,Non Heinous,Pending Trial,Written,Karnataka State Local Act,...,1,0,2123,2016-10-27,Thursday,0,4,1317,7717,1317_7717
1,Bengaluru City,Chickpet Traffic PS,2017,2,0,7,Non Heinous,Convicted,Written,PUBLIC SAFETY,...,1,0,1788,2017-02-07,Tuesday,0,1,1296,7756,1296_7756
2,Bengaluru City,Puttenahalli PS,2018,6,0,6,Heinous,Pending Trial,Written,BURGLARY - DAY,...,1,0,2203,2018-06-06,Wednesday,0,2,1288,7757,1288_7757
3,Bengaluru City,South CEN Crime PS,2020,8,0,24,Non Heinous,Compounded,Written,CYBER CRIME,...,1,0,2258,2020-08-24,Monday,0,3,1273,7801,1273_7801
4,Bengaluru City,Byatarayanapura PS,2021,12,22,15,Non Heinous,Pending Trial,Oral,CRIMES RELATED TO WOMEN,...,4,0,1418,2021-12-15,Wednesday,0,4,1294,7753,1294_7753


In [7]:
df["Date"] = pd.to_datetime(df["Date"])

print(df["Date"].min())
print(df["Date"].max())

2016-01-01 00:00:00
2023-12-31 00:00:00


In [8]:
daily_crime = (
    df.groupby(["Grid_ID", "Date"])
      .size()
      .reset_index(name="Crime_Count")
)

daily_crime.head()

,Grid_ID,Date,Crime_Count
0,100_7761,2017-09-14,1
1,1033_7799,2023-08-31,1
2,1074_7911,2020-02-02,1
3,1079_7878,2020-02-05,1
4,1115_7907,2023-01-19,1


In [9]:
print(daily_crime["Crime_Count"].describe())

count    117425.000000
mean          1.258412
std           0.793144
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          25.000000
Name: Crime_Count, dtype: float64


In [20]:
# Convert Date to datetime
df["Date"] = pd.to_datetime(df["Date"])

# Create Week Number
df["Week"] = df["Date"].dt.isocalendar().week
df["Year"] = df["Date"].dt.year

# Weekly crime count
weekly_crime = (
    df.groupby(["Grid_ID", "UnitName", "Year", "Week"])
      .size()
      .reset_index(name="Crime_Count")
)

weekly_crime.head()

,Grid_ID,UnitName,Year,Week,Crime_Count
0,100_7761,Hulimavu Traffic PS,2017,37,1
1,1033_7799,Basaveshwara Nagar PS,2023,35,1
2,1074_7911,Jayanagar PS,2020,5,1
3,1079_7878,Byadarahalli PS,2020,6,1
4,1115_7907,Kempapura Agrahara PS,2023,3,1


In [21]:
print(weekly_crime["Crime_Count"].describe())

count    74818.000000
mean         1.975046
std          2.484829
min          1.000000
25%          1.000000
50%          1.000000
75%          2.000000
max         80.000000
Name: Crime_Count, dtype: float64


In [22]:
weekly_crime["Crime_Count"].value_counts().sort_index()

,count
Crime_Count,
1,47185
2,12686
3,6085
4,3378
5,1896
...,...
71,1
74,1
76,1


In [23]:
def assign_risk(count):
    if count == 1:
        return "Low"
    elif count <= 3:
        return "Medium"
    else:
        return "High"

weekly_crime["Risk_Label"] = weekly_crime["Crime_Count"].apply(assign_risk)

weekly_crime.head()

,Grid_ID,UnitName,Year,Week,Crime_Count,Risk_Label
0,100_7761,Hulimavu Traffic PS,2017,37,1,Low
1,1033_7799,Basaveshwara Nagar PS,2023,35,1,Low
2,1074_7911,Jayanagar PS,2020,5,1,Low
3,1079_7878,Byadarahalli PS,2020,6,1,Low
4,1115_7907,Kempapura Agrahara PS,2023,3,1,Low


In [24]:
print(weekly_crime["Risk_Label"].value_counts())

Risk_Label
Low       47185
Medium    18771
High       8862
Name: count, dtype: int64


In [26]:
weekly_crime.to_csv("/content/drive/MyDrive/AI_Crime_Prediction/datasets/Weekly_Risk_Dataset.csv", index=False)

print("Weekly Risk Dataset Saved!")

Weekly Risk Dataset Saved!


In [19]:
import pandas as pd

# Load the file again
station_master = pd.read_csv("/content/drive/MyDrive/AI_Crime_Prediction/metadata/PoliceStation_Master.csv")

# Print the columns to see what they are actually named
print("Columns in station_master:", station_master.columns.tolist())

Columns in station_master: ['UnitName', 'Unit_ID', 'Average_Latitude', 'Average_Longitude', 'Total_FIRs']
